In [3]:
from typing import Dict, List
import copy


heuristics = {
'r':5,'n':3,'b':3,'q':9,'k':100,'p':1,
'R':5,'N':3,'B':3,'Q':9,'K':100,'P':1
}

board = [
['r','n','b','q','k','b','n','r'],
['p','p','p','p','p','p','p','p'],
['.','.','.','.','.','.','.','.'],
['.','.','.','.','.','.','.','.'],
['.','.','.','.','.','.','.','.'],
['.','.','.','.','.','.','.','.'],
['P','P','P','P','P','P','P','P'],
['R','N','B','Q','K','B','N','R']
]


class Move:

    def __init__(self, start, end, score, piece):
        self.start = start 
        self.end   = end
        self.score = score
        self.piece = piece 
        
    def __lt__(self, other):
        return self.score < other.score

    def __str__(self):
        return f"{self.piece}: {self.start} -> {self.end}"


def evalulate_board(board):

    score = 0

    for i in range(8):
        for j in range(8):

            piece = board[i][j]

            if piece == '.':
                continue

            if piece.isupper():
                score += heuristics[piece]
            else:
                score -= heuristics[piece]

    return score


def apply_move(board, move):

    new_board = copy.deepcopy(board)

    r1, c1 = move.start
    r2, c2 = move.end
    
    new_board[r2][c2] = new_board[r1][c1]
    new_board[r1][c1] = '.'

    return new_board



def get_sliding_move(piece, board, dirs, i, j):

    moves = []

    for dx, dy in dirs:

        x = i + dx
        y = j + dy

        while 0 <= x < 8 and 0 <= y < 8:

            target = board[x][y]

            if target == '.':
                moves.append(Move((i,j),(x,y),0,piece))

            elif target.islower():

                moves.append(
                    Move((i,j),(x,y),heuristics[target],piece)
                )

                break

            else:
                break

            x += dx
            y += dy

    return moves
    


def get_possible_moves(board):

    moves = []

    knight_moves=[(-2,-1),(-2,1),(-1,-2),(-1,2),
                  (1,-2),(1,2),(2,-1),(2,1)]

    king_moves=[(-1,-1),(-1,0),(-1,1),
                (0,-1),(0,1),
                (1,-1),(1,0),(1,1)]

    rook_dirs=[(-1,0),(1,0),(0,-1),(0,1)]

    bishop_dirs=[(-1,-1),(-1,1),(1,-1),(1,1)]

    queen_dirs=rook_dirs + bishop_dirs


    for i in range(8):
        for j in range(8):

            piece = board[i][j]

            if piece == '.' or piece.islower():
                continue


            # Pawn
            if piece == 'P':

                if i-1 >= 0 and board[i-1][j] == '.':
                    moves.append(
                        Move((i,j),(i-1,j),0,piece)
                    )

                for dy in [-1,1]:

                    if 0 <= j+dy < 8 and i-1 >= 0:

                        target = board[i-1][j+dy]

                        if target.islower():

                            moves.append(
                                Move((i,j),
                                (i-1,j+dy),
                                heuristics[target],
                                piece)
                            )


            # Knight
            if piece == 'N':

                for dx,dy in knight_moves:

                    x = i + dx
                    y = j + dy

                    if 0 <= x < 8 and 0 <= y < 8:

                        target = board[x][y]

                        if target == '.' or target.islower():

                            score = 0

                            if target != '.':
                                score = heuristics[target]

                            moves.append(
                                Move((i,j),(x,y),score,piece)
                            )


            # King
            if piece == 'K':

                for dx,dy in king_moves:

                    x = i + dx
                    y = j + dy

                    if 0 <= x < 8 and 0 <= y < 8:

                        target = board[x][y]

                        if target == '.' or target.islower():

                            score = 0

                            if target != '.':
                                score = heuristics[target]

                            moves.append(
                                Move((i,j),(x,y),score,piece)
                            )


            if piece == 'B':
                moves += get_sliding_move(piece,board,bishop_dirs,i,j)

            if piece == 'R':
                moves += get_sliding_move(piece,board,rook_dirs,i,j)

            if piece == 'Q':
                moves += get_sliding_move(piece,board,queen_dirs,i,j)


    return moves



def beam_search(board, depth, beam_width):

    beam = [([], board, evalulate_board(board))]

    for _ in range(depth):

        candidates = []

        for path, state, score in beam:

            moves = get_possible_moves(state)
            
            for m in moves:

                new_state = apply_move(state, m)

                new_path = path + [m]

                new_score = evalulate_board(new_state)

                candidates.append(
                    (new_path,new_state,new_score)
                )


        candidates.sort(key=lambda x: x[2], reverse=True)

        beam = candidates[:beam_width]


    return beam[0]



best_path, board_state, score = beam_search(board,3,3)


print("Best path:")
for move in best_path:
    print(move)

print("\nScore:",score)

Best path:
P: (6, 0) -> (5, 0)
P: (5, 0) -> (4, 0)
P: (4, 0) -> (3, 0)

Score: 0


In [171]:
from typing import Dict, List, Tuple

from typing import Dict, Tuple
# data was gernerated by AI, to make things interesting 
# CPU jobs with (execution_time, priority)
Jobs: Dict[str, Tuple[int,int]] = {
    'CompileCode': (12, 3),
    'RenderVideo': (27, 30),
    'DatabaseQuery': (7, 9),
    'BackupFiles': (9, 1),
    'SystemScan': (4, 2),
    'ProcessImages': (15, 10),
    'EncryptData': (8, 7),
    'AITraining': (20, 15),
    'NetworkSync': (3, 6),
    'CacheCleanup': (5, 3),
    'AudioProcessing': (10, 12),
    'LogAnalysis': (6, 4),
    'VideoStreaming': (18, 20),
    'TaskScheduler': (2, 8),
    'LoadBalancer': (1, 5)
}



def beam_search(Jobs, beam_width: int):

    beam = [([],0,set(Jobs.keys()))]
    while beam:
        candidates = []
        for path, score, remaining in beam:
            if not remaining:
                return (path,score)
            for job_name in remaining:
               
                exec_time, priority = Jobs[job_name]        
                new_path = path + [job_name]
                new_score = score + abs(exec_time-priority)
                new_remaining = remaining.copy()
                new_remaining.remove(job_name)

                candidates.append((new_path,new_score,new_remaining))


        candidates.sort(key=lambda x:x[1], reverse=True)

        beam = candidates[:beam_width]

    return beam[0]



job_path = beam_search(Jobs,2)

print(job_path)

(['CompileCode', 'BackupFiles', 'TaskScheduler', 'AITraining', 'ProcessImages', 'LoadBalancer', 'NetworkSync', 'RenderVideo', 'AudioProcessing', 'VideoStreaming', 'SystemScan', 'CacheCleanup', 'DatabaseQuery', 'LogAnalysis', 'EncryptData'], 56)


In [ ]:
from random import choice, triangular 
import random 
from typing import List 
# data was gernerated by AI, to make things interesting 
graph = {
    'Times Square': [('Central Park',3), ('Wall Street',6), ('Brooklyn Bridge',5), ('Grand Central',4)],
    'Central Park': [('Columbia University',9), ('Museum of Modern Art',8), ('Lincoln Center',7)],
    'Wall Street': [('Battery Park',12), ('One World Trade Center',14), ('South Street Seaport',6)],
    'Brooklyn Bridge': [('DUMBO',7), ('City Hall',3)],
    'DUMBO': [('Williamsburg',5), ('Prospect Park',6), ('Brooklyn Heights',4)],
    'Williamsburg': [('Bushwick',1), ('Queens',10), ('Coney Island',2)],
    'Columbia University': [('Morningside Heights',3)],
    'Museum of Modern Art': [('Fifth Avenue',2)],
    'Battery Park': [('Staten Island Ferry',5)],
    'One World Trade Center': [('World Trade Center Oculus',1)],
    'Prospect Park': [('Brooklyn Botanic Garden',2)],
    'Bushwick': [('Bedford-Stuyvesant',3)],
    'Queens': [('Flushing Meadows',4)],
    'Coney Island': [('Brighton Beach',1)],
    'Grand Central': [('Chrysler Building',1)],
    'Lincoln Center': [],
    'South Street Seaport': [],
    'City Hall': [],
    'Brooklyn Heights': [],
    'Morningside Heights': [],
    'Fifth Avenue': [],
    'Staten Island Ferry': [],
    'World Trade Center Oculus': [],
    'Brooklyn Botanic Garden': [],
    'Bedford-Stuyvesant': [],
    'Flushing Meadows': [],
    'Brighton Beach': [],
    'Chrysler Building': []
}

traffic_density = {
    'Times Square': 10,
    'Central Park': 9,
    'Wall Street': 7,
    'Brooklyn Bridge': 5,
    'Columbia University': 8,
    'Museum of Modern Art': 6,
    'Battery Park': 4,
    'One World Trade Center': 3,
    'DUMBO': 3,
    'Williamsburg': 2,
    'Prospect Park': 6,
    'Bushwick': 2,
    'Queens': 1,
    'Coney Island': 2,
    'Grand Central': 7,
    'Lincoln Center': 5,
    'South Street Seaport': 4,
    'City Hall': 3,
    'Brooklyn Heights': 2,
    'Morningside Heights': 4,
    'Fifth Avenue': 3,
    'Staten Island Ferry': 5,
    'World Trade Center Oculus': 1,
    'Brooklyn Botanic Garden': 2,
    'Bedford-Stuyvesant': 3,
    'Flushing Meadows': 2,
    'Brighton Beach': 1,
    'Chrysler Building': 4
}

def hill_climbing(graph, start, goal):
    current_node = start
    path = [current_node]
    tolerance_factor = 20
    while current_node != goal:
         neighbours = graph.get(current_node, [])
         random_node = random.choice(list(traffic_density.keys()))
         traffic_density[random_node] = random.randint(1, 7)
         if not neighbours:
            return None
         best_neighbour = min(neighbours, key=lambda x : traffic_density[x[0]])
         if traffic_density[best_neighbour[0]] >= traffic_density[current_node] + tolerance_factor:
                return None
         current_node = best_neighbour[0]
         path.append(current_node)
          
         if current_node == goal:
            return path 
    return None


random_destination = random.choice(list(graph.keys())) # to simulate multipule deliveries 
Initial_state = 'Times Square'
path = None


path = hill_climbing(graph,Initial_state , random_destination)
print(f" path for delivery from {Initial_state} to {random_destination} : {path}")






 path for delivery from Times Square to Flushing Meadows : ['Times Square', 'Brooklyn Bridge', 'DUMBO', 'Williamsburg', 'Queens', 'Flushing Meadows']


In [ ]:
import random
import math
# got the Data and longitude and latitude forumlas from AI to make things interesting and detailed
# --- Step 1: Define 10 UK cities with approximate lat/lon ---
cities = {
    0: ("London", (51.5074, -0.1278)),
    1: ("Manchester", (53.4808, -2.2426)),
    2: ("Birmingham", (52.4862, -1.8904)),
    3: ("Liverpool", (53.4084, -2.9916)),
    4: ("Leeds", (53.8008, -1.5491)),
    5: ("Glasgow", (55.8642, -4.2518)),
    6: ("Edinburgh", (55.9533, -3.1883)),
    7: ("Bristol", (51.4545, -2.5879)),
    8: ("Cardiff", (51.4816, -3.1791)),
    9: ("Newcastle", (54.9783, -1.6178))
}

NUM_CITIES = len(cities)

# --- Step 2: Haversine distance between two cities ---
def haversine(coord1, coord2):
    R = 6371  # Earth radius in km
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    phi1 = math.radians(lat1) t
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    
    a = math.sin(delta_phi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# --- Step 3: Fitness function (total route distance) ---
def fitness(route):
    total_dist = 0
    for i in range(NUM_CITIES):
        city1 = route[i]
        city2 = route[(i + 1) % NUM_CITIES]
        total_dist += haversine(cities[city1][1], cities[city2][1])
    return total_dist

# --- Step 4: Initialize population ---
def create_population(pop_size):
    population = []
    for _ in range(pop_size):
        route = list(cities.keys())
        random.shuffle(route)
        population.append(route)
    return population

# --- Step 5: Selection (tournament selection) ---
def selection(population, fitnesses, k=3):
    selected = []
    for _ in range(len(population)):
        candidates = random.sample(list(zip(population, fitnesses)), k)
        candidates.sort(key=lambda x: x[1])  # lower distance is better
        selected.append(candidates[0][0])
    return selected

# --- Step 6: Crossover (ordered crossover) ---
def crossover(parent1, parent2):
    start, end = sorted(random.sample(range(NUM_CITIES), 2))
    child = [None] * NUM_CITIES
    child[start:end+1] = parent1[start:end+1]
    
    pointer = 0
    for city in parent2:
        if city not in child:
            while child[pointer] is not None:
                pointer += 1
            child[pointer] = city
    return child

# --- Step 7: Mutation (swap mutation) ---
def mutate(route, mutation_rate=0.1):
    for i in range(NUM_CITIES):
        if random.random() < mutation_rate:
            j = random.randint(0, NUM_CITIES-1)
            route[i], route[j] = route[j], route[i]
    return route

# --- Step 8: Genetic Algorithm ---
def genetic_algorithm(pop_size=100, generations=500, mutation_rate=0.1):
    population = create_population(pop_size)
    
    for gen in range(generations):
        fitnesses = [fitness(route) for route in population]
        best_idx = min(range(len(fitnesses)), key=lambda i: fitnesses[i])
        best_route = population[best_idx]
        best_distance = fitnesses[best_idx]
        
        if gen % 50 == 0:
            print(f"Generation {gen}: Best distance = {best_distance:.2f} km")
        
        selected = selection(population, fitnesses)
        next_population = []
        for i in range(0, pop_size, 2):
            parent1 = selected[i]
            parent2 = selected[i+1 if i+1 < pop_size else 0]
            child1 = crossover(parent1, parent2)
            child2 = crossover(parent2, parent1)
            next_population.append(mutate(child1, mutation_rate))
            next_population.append(mutate(child2, mutation_rate))
        
        population = next_population
    
    fitnesses = [fitness(route) for route in population]
    best_idx = min(range(len(fitnesses)), key=lambda i: fitnesses[i])
    return population[best_idx], fitnesses[best_idx]

# --- Run the GA ---
best_route, best_distance = genetic_algorithm()
print("\nBest Route:")
for city in best_route:
    print(cities[city][0], end=" -> ")
print(cities[best_route[0]][0])  # return to start
print(f"Total Distance: {best_distance:.2f} km")

Generation 0: Best distance = 1952.10 km
Generation 50: Best distance = 1499.24 km
Generation 100: Best distance = 1445.76 km
Generation 150: Best distance = 1389.33 km
Generation 200: Best distance = 1445.76 km
Generation 250: Best distance = 1488.37 km
Generation 300: Best distance = 1549.41 km
Generation 350: Best distance = 1445.76 km
Generation 400: Best distance = 1445.76 km
Generation 450: Best distance = 1496.55 km

Best Route:
Cardiff -> Bristol -> London -> Birmingham -> Manchester -> Liverpool -> Glasgow -> Edinburgh -> Newcastle -> Leeds -> Cardiff
Total Distance: 1447.37 km
